In [ ]:
from helptools import register_user, login_user, identify
from PersonsDataset import PersonDataset
from dotenv import load_dotenv
from faker import Faker
import itertools as it
import random
import tqdm

In [ ]:
load_dotenv()

In [ ]:
train_dataset = PersonDataset("../data/vggface2/train")

In [ ]:
fake = Faker()
subset = list(it.islice(train_dataset, 250))
train_check = {}
for images, person_id in tqdm.tqdm(subset):
    first_name = fake.unique.first_name()
    last_name = fake.unique.last_name()

    response = await register_user(
        images=images, first_name=first_name, last_name=last_name
    )

    id = response.json().get("id")
    if id is not None:
        train_check[id] = images[:2]

In [ ]:
FR = 0  # False reject
FA = 0  # False acceptance
TA = 0  # True acceptance
TR = 0  # True reject

In [ ]:
ids = train_check.keys()

for id, images in train_check.items():
    for img in images:
        r = await login_user(img, id)
        s = r.json().get('success')
        if s is not None:
            if s:
                TA += 1
            else:
                FR += 1

        other_id = random.choice(ids)
        while other_id == id:
            other_id = random.choice(ids)

        r = await login_user(img, other_id)
        s = r.json().get('success')
        if s is not None:
            if s:
                FA += 1
            else:
                TR += 1

**ZAD 1**

dodanie 200 osób do systemu

FAR i FRR na 400 przykładach sprawdzenie i autoryzacja do siebie i próba ataku

In [ ]:
f"FAR: {FA/(FA+TR)} \n FRR:{FR/(FR+TA)}"

In [ ]:
val_dataset = PersonDataset("../data/vggface2/val")

In [ ]:
fake = Faker()
subset = list(it.islice(train_dataset, 60))
possible_attacks = 0
for images, person_id in tqdm.tqdm(subset):

    responses = []

    for image in images[:2]:
        responses += [await identify(image) for image in images]

    contents = list(map(lambda resp: resp.json().get("found", []), responses))

    possible_attacks += len(set(sum(contents, [])))

**ZAD 2 dodatek**

ataki przy autoryzacji 1:N

In [ ]:
possible_attacks

In [ ]:
A = 0
R = 0

In [ ]:
ids = train_check.keys()

for images, person_id in tqdm.tqdm(subset):
    for img in images:

        id = random.choice(ids)

        r = await login_user(img, id)
        s = r.json().get('success')
        if s is not None:
            if s:
                A += 1
            else:
                R += 1

**ZAD 2**

120 prób autoryzacji zdjęciami osób z poza puli treningowej

In [ ]:
f"{A} \n {R} \n {FA+A/(FA+A+R+TR)}"

In [ ]:
import cv2
import numpy as np

In [ ]:
def add_gaussian_noise_psnr(img_path, psnr_pairs=[(50,80),(40,50),(30,40),(20,30),(10,20)]):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    def psnr_to_std(img, target_psnr):
        mse = np.mean(img.astype(np.float32)**2) / (10**(target_psnr/10))
        return np.sqrt(mse)

    noisy_images = {}
    img_float = img.astype(np.float32)

    for (psnr_min, psnr_max) in psnr_pairs:
        target_psnr = np.random.uniform(psnr_min, psnr_max)
        std = psnr_to_std(img_float, target_psnr)
        
        gauss = np.random.normal(0, std, img.shape).astype(np.float32)
        noisy_img = img_float + gauss
        noisy_img = np.clip(noisy_img, 0, 255).astype(np.uint8)
        
        noisy_images[(psnr_min, psnr_max)] = noisy_img

    return noisy_images

In [ ]:
base_dict = {(50,80): 0,(40,50): 0,(30,40): 0,(20,30): 0,(10,20): 0}
FR_noise = base_dict.copy()  # False reject
FA_noise = base_dict.copy()  # False acceptance
TA_noise = base_dict.copy()  # True acceptance
TR_noise = base_dict.copy()  # True reject

In [ ]:
for id, images in train_check.items():

    images = [add_gaussian_noise_psnr(i) for i in images]
    
    for imgs in images:
        for k, img in imgs.items():
            r = await login_user(img, id)
            s = r.json().get('success')
            if s is not None:
                if s:
                    TA_noise[k] += 1
                else:
                    FR_noise[k] += 1

            other_id = random.choice(ids)
            while other_id == id:
                other_id = random.choice(ids)

            r = await login_user(img, other_id)
            s = r.json().get('success')
            if s is not None:
                if s:
                    FA_noise[k] += 1
                else:
                    TR_noise[k] += 1

**ZAD 3**

porównanie wpływu szumu o różnej mocy na wyniki

In [ ]:
for k in base_dict.keys():
    print(f"k \n FAR: {FA_noise[k]/(FA_noise[k]+TR_noise[k])} \n FRR:{FR_noise[k]/(FR_noise[k]+TA_noise[k])} \n")

In [ ]:
def change_luminance(img_path):
    img = cv2.imread(img_path)
    img_ycrcb = cv2.cvtColor(img, cv2.COLOR_BGR2YCrCb).astype(np.float32)
    Y, Cr, Cb = cv2.split(img_ycrcb)
    
    results = {}
    
    scale_factors = [0.5, 0.6, 0.75, 1.33, 1.5]
    for factor in scale_factors:
        Y_new = np.clip(Y * factor, 0, 255)
        img_new = cv2.merge([Y_new, Cr, Cb]).astype(np.uint8)
        img_rgb = cv2.cvtColor(img_new, cv2.COLOR_YCrCb2RGB)
        results[f"scale_{factor:.2f}"] = img_rgb
    
    add_values = [-100, -20, -10, 30]
    for val in add_values:
        Y_new = np.clip(Y + val, 0, 255)
        img_new = cv2.merge([Y_new, Cr, Cb]).astype(np.uint8)
        img_rgb = cv2.cvtColor(img_new, cv2.COLOR_YCrCb2RGB)
        results[f"add_{val}"] = img_rgb
    
    return results

In [ ]:
base_dict = {f"scale_{factor:.2f}": 0 for factor in [0.5, 0.6, 0.75, 1.33, 1.5]}
base_dict += {f"add_{val}": 0 for val in [-100, -20, -10, 30]}
FR_scale_y = base_dict.copy()  # False reject
FA_scale_y = base_dict.copy()  # False acceptance
TA_scale_y = base_dict.copy()  # True acceptance
TR_scale_y = base_dict.copy()  # True reject

In [ ]:
for id, images in train_check.items():

    images = [change_luminance(i) for i in images]
    
    for imgs in images:
        for k, img in imgs.items():
            r = await login_user(img, id)
            s = r.json().get('success')
            if s is not None:
                if s:
                    TA_scale_y[k] += 1
                else:
                    FR_scale_y[k] += 1

            other_id = random.choice(ids)
            while other_id == id:
                other_id = random.choice(ids)

            r = await login_user(img, other_id)
            s = r.json().get('success')
            if s is not None:
                if s:
                    FA_scale_y[k] += 1
                else:
                    TR_scale_y[k] += 1

**ZAD 4**

porównanie wpływu zmiany luminacji na wyniki

In [ ]:
for k in base_dict.keys():
    print(f"FAR: {FA_scale_y[k]/(FA_scale_y[k]+TR_scale_y[k])} \n FRR:{FR_scale_y[k]/(FR_scale_y[k]+TA_scale_y[k])}")